# 4.6 Rank of a Matrix and Systems of Linear Equations

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4 — Vector Spaces**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply, transpose, add, scalar_mul
from linalg_core.elimination import solve, to_rref
from linalg_core.vecspace import (
    rank, nullity, null_space, column_space, row_space, is_independent
)
from linalg_core import EPSILON
import numpy as np

print("Setup complete.")

---

## 2. Motivation

Every system of linear equations $Ax = b$ falls into exactly one of three categories:
it has a **unique solution**, **infinitely many solutions**, or **no solution at all**.

**Rank is the single number that explains which case you’re in.**

Given an $m \times n$ matrix $A$ and a vector $b \in \mathbb{R}^m$, the rank of $A$
tells us:

| Condition | Result |
|---|---|
| $\operatorname{rank}(A) = \operatorname{rank}([A \mid b]) = n$ | Unique solution |
| $\operatorname{rank}(A) = \operatorname{rank}([A \mid b]) < n$ | Infinitely many solutions |
| $\operatorname{rank}(A) < \operatorname{rank}([A \mid b])$ | No solution (inconsistent) |

To understand *why* this works, we need to study three fundamental subspaces
attached to every matrix: the **row space**, **column space**, and **null space**.
Their dimensions are tied together by the **Rank-Nullity Theorem**, one of the
most important results in linear algebra.

---

## 3. Build — The Three Fundamental Subspaces

Every $m \times n$ matrix $A$ defines three subspaces. We build intuition for
each one, then connect them through rank.

### 3.1 Row Space

The **row space** of $A$ is the subspace of $\mathbb{R}^n$ spanned by the rows of $A$:

$$\operatorname{Row}(A) = \operatorname{span}\{\text{row}_1, \text{row}_2, \ldots, \text{row}_m\}$$

**Key fact:** Row operations do not change the row space. So the nonzero rows of the
RREF give a basis for $\operatorname{Row}(A)$.

`row_space(A)` returns these nonzero rows of the RREF.

In [ ]:
A = Matrix.from_lists([
    [1, 2, 3, 4],
    [2, 4, 7, 10],
    [3, 6, 10, 14]
])

print("A ="); print(A)
print()

R = A.copy()
pivots = to_rref(R)
print("RREF of A ="); print(R)
print(f"\nPivot positions: {pivots}")

rs = row_space(A)
print(f"\nRow space basis ({len(rs)} vectors):")
for i, v in enumerate(rs):
    print(f"  r{i+1} = {v}")
print(f"\nDimension of row space = {len(rs)}")

### 3.2 Column Space

The **column space** of $A$ is the subspace of $\mathbb{R}^m$ spanned by the columns of $A$:

$$\operatorname{Col}(A) = \operatorname{span}\{\text{col}_1, \text{col}_2, \ldots, \text{col}_n\}$$

Equivalently, $\operatorname{Col}(A) = \{Ax : x \in \mathbb{R}^n\}$ — it is the set of all
possible outputs of the linear transformation $x \mapsto Ax$.

**Key fact:** The pivot columns of the RREF tell us *which* original columns of $A$ form
a basis. We take the **original** columns (not the RREF columns!) at those positions.

`column_space(A)` returns the original columns at pivot positions.

In [ ]:
print("A ="); print(A)
print()

cs = column_space(A)
print(f"Column space basis ({len(cs)} vectors):")
for i, v in enumerate(cs):
    print(f"  c{i+1} = {v}")
print(f"\nDimension of column space = {len(cs)}")

print("\n--- Why original columns, not RREF columns? ---")
print("RREF columns at pivot positions:")
R = A.copy()
pivot_positions = to_rref(R)
pivot_cols = sorted(col for _, col in pivot_positions)
for col_idx in pivot_cols:
    print(f"  RREF col {col_idx} = {R.get_col(col_idx)}")
print("\nOriginal columns at pivot positions:")
for col_idx in pivot_cols:
    print(f"  A col {col_idx} = {A.get_col(col_idx)}")
print("\nRREF columns are just standard basis vectors — they span a different subspace!")
print("The original columns give the actual column space of A.")

### 3.3 Null Space

The **null space** (or **kernel**) of $A$ is the set of all solutions to the homogeneous system $Ax = 0$:

$$\operatorname{Null}(A) = \{x \in \mathbb{R}^n : Ax = 0\}$$

This is always a subspace of $\mathbb{R}^n$. Each free variable in the RREF produces
one basis vector for the null space.

`null_space(A)` returns basis vectors, one per free variable.

In [ ]:
print("A ="); print(A)
print()

ns = null_space(A)
print(f"Null space basis ({len(ns)} vectors):")
for i, v in enumerate(ns):
    print(f"  n{i+1} = {[round(x, 6) for x in v]}")
print(f"\nDimension of null space = {len(ns)}")

print("\n--- Verification: each null space vector satisfies Ax = 0 ---")
for i, v in enumerate(ns):
    x = Matrix.from_vector(v)
    Ax = multiply(A, x)
    result = [Ax[j, 0] for j in range(Ax.rows)]
    is_zero = all(abs(val) < EPSILON for val in result)
    print(f"  A * n{i+1} = {[round(val, 10) for val in result]}  (zero? {is_zero})")

### 3.4 Rank — The Connecting Number

The **rank** of $A$ is defined as the number of pivot positions in the RREF.
A remarkable theorem states that this equals both:

$$\operatorname{rank}(A) = \dim(\operatorname{Row}(A)) = \dim(\operatorname{Col}(A)) = \text{number of pivots}$$

This is not obvious — the row space lives in $\mathbb{R}^n$ and the column space lives in
$\mathbb{R}^m$, yet they always have the same dimension. This is because row operations
preserve the linear dependencies among columns.

`rank(A)` returns this number.

In [ ]:
matrices = {
    "Full rank (3x3)": Matrix.from_lists([[1, 0, 1], [0, 1, 1], [1, 1, 0]]),
    "Rank 2 (3x3)": Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    "Rank 2 (3x4)": A,
    "Rank 1 (2x3)": Matrix.from_lists([[1, 2, 3], [2, 4, 6]]),
    "Identity (3x3)": Matrix.identity(3),
}

for name, M in matrices.items():
    r = rank(M)
    rs_dim = len(row_space(M))
    cs_dim = len(column_space(M))
    print(f"{name}:")
    print(f"  rank = {r}, dim(Row) = {rs_dim}, dim(Col) = {cs_dim}")
    print(f"  All three equal? {r == rs_dim == cs_dim}")
    print()

### 3.5 Nullity — Dimension of the Null Space

The **nullity** of $A$ is the dimension of its null space:

$$\operatorname{nullity}(A) = \dim(\operatorname{Null}(A)) = \text{number of free variables}$$

Since each column is either a pivot column or a free column, we have:

$$\operatorname{nullity}(A) = n - \operatorname{rank}(A)$$

where $n$ is the number of columns.

`nullity(A)` returns this number.

In [ ]:
for name, M in matrices.items():
    r = rank(M)
    nu = nullity(M)
    ns_dim = len(null_space(M))
    print(f"{name} ({M.rows}x{M.cols}):")
    print(f"  rank = {r}, nullity = {nu}, dim(Null) = {ns_dim}")
    print(f"  nullity == dim(Null)? {nu == ns_dim}")
    print(f"  nullity == cols - rank? {nu == M.cols - r}")
    print()

### 3.6 The Rank-Nullity Theorem

**Theorem (Rank-Nullity).** For any $m \times n$ matrix $A$:

$$\boxed{\operatorname{rank}(A) + \operatorname{nullity}(A) = n}$$

where $n$ is the number of columns of $A$.

**Why it works:** Every column of $A$ is either a pivot column (contributes to rank)
or a free column (contributes to nullity). There are exactly $n$ columns total.

This theorem says that the "information" in $A$ splits into two complementary parts:
- The rank measures how many independent directions $A$ maps **to** (the image).
- The nullity measures how many independent directions $A$ collapses **to zero** (the kernel).

Together they always account for all $n$ input dimensions.

In [ ]:
print("=" * 60)
print("RANK-NULLITY THEOREM: rank(A) + nullity(A) = n (columns)")
print("=" * 60)
print()

test_matrices = [
    ("3x4 matrix", Matrix.from_lists([[1, 2, 3, 4], [2, 4, 7, 10], [3, 6, 10, 14]])),
    ("3x3 rank-deficient", Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 9]])),
    ("3x3 full rank", Matrix.from_lists([[1, 0, 1], [0, 1, 1], [1, 1, 0]])),
    ("2x3 rank 1", Matrix.from_lists([[1, 2, 3], [2, 4, 6]])),
    ("4x4 identity", Matrix.identity(4)),
    ("2x5 matrix", Matrix.from_lists([[1, 0, 2, 0, 1], [0, 1, 0, 3, 2]])),
    ("3x3 zero matrix", Matrix(3, 3)),
    ("1x4 row vector", Matrix.from_lists([[3, 1, 4, 1]])),
]

all_pass = True
for name, M in test_matrices:
    r = rank(M)
    nu = nullity(M)
    n = M.cols
    holds = (r + nu == n)
    if not holds:
        all_pass = False
    print(f"{name} ({M.rows}x{M.cols}): rank={r}, nullity={nu}, n={n}  |  {r}+{nu}={r+nu}=={n}? {holds}")

print(f"\nRank-Nullity holds for all test matrices: {all_pass}")

### 3.7 Rank and Solution Types

Now we connect rank to the solvability of $Ax = b$. Given the augmented matrix $[A \mid b]$:

| Condition | Interpretation | Solution type |
|---|---|---|
| $\operatorname{rank}(A) = \operatorname{rank}([A \mid b]) = n$ | $b \in \operatorname{Col}(A)$, no free vars | **Unique** |
| $\operatorname{rank}(A) = \operatorname{rank}([A \mid b]) < n$ | $b \in \operatorname{Col}(A)$, free vars exist | **Infinite** |
| $\operatorname{rank}(A) < \operatorname{rank}([A \mid b])$ | $b \notin \operatorname{Col}(A)$ | **Inconsistent** |

**Why?**
- If $\operatorname{rank}(A) = \operatorname{rank}([A \mid b])$, the last column of the
  augmented matrix does not introduce a new pivot — meaning $b$ is a linear combination
  of the columns of $A$.
- If additionally $\operatorname{rank}(A) = n$, there are no free variables, so the
  representation is unique.
- If $\operatorname{rank}(A) < \operatorname{rank}([A \mid b])$, the augmented matrix has a
  pivot in the last column, giving a row like $[0 \cdots 0 \mid c]$ with $c \neq 0$,
  which is a contradiction $0 = c$.

In [ ]:
def analyze_system(A, b, label=""):
    """Analyze Ax=b using rank conditions and verify with solve()."""
    n = A.cols
    aug = Matrix(A.rows, A.cols + 1)
    for i in range(A.rows):
        for j in range(A.cols):
            aug[i, j] = A[i, j]
        aug[i, A.cols] = b[i]

    r_A = rank(A)
    r_aug = rank(aug)

    if r_A == r_aug == n:
        predicted = "Unique"
    elif r_A == r_aug < n:
        predicted = "Infinite"
    else:
        predicted = "Inconsistent"

    sol_type, result = solve(aug)
    actual = sol_type.capitalize()

    match = predicted.lower() == actual.lower()

    if label:
        print(f"--- {label} ---")
    print(f"  A is {A.rows}x{A.cols}, rank(A)={r_A}, rank([A|b])={r_aug}, n={n}")
    print(f"  Predicted: {predicted}")
    print(f"  Actual:    {actual}")
    if sol_type == "unique":
        print(f"  Solution:  x = {[round(v, 6) for v in result]}")
    elif sol_type == "infinite":
        print(f"  Particular: {[round(v, 6) for v in result['particular']]}")
        print(f"  Free vars:  {result['free_vars']}")
    print(f"  Rank prediction matches solve()? {match}")
    print()
    return match


print("=" * 60)
print("RANK AND SOLUTION TYPES")
print("=" * 60)
print()

A1 = Matrix.from_lists([[1, 2], [3, 5]])
b1 = [5, 13]
m1 = analyze_system(A1, b1, "Case 1: Unique (rank=rank_aug=n)")

A2 = Matrix.from_lists([[1, 2, 3], [2, 4, 6], [1, 2, 3]])
b2 = [1, 2, 1]
m2 = analyze_system(A2, b2, "Case 2: Infinite (rank=rank_aug<n)")

A3 = Matrix.from_lists([[1, 2], [3, 6]])
b3 = [1, 4]
m3 = analyze_system(A3, b3, "Case 3: Inconsistent (rank<rank_aug)")

A4 = Matrix.from_lists([[1, 0, 1], [0, 1, 1], [1, 1, 0]])
b4 = [3, 4, 3]
m4 = analyze_system(A4, b4, "Case 4: 3x3 unique")

A5 = Matrix.from_lists([[1, 2, 3, 4], [2, 4, 7, 10], [3, 6, 10, 14]])
b5 = [10, 23, 33]
m5 = analyze_system(A5, b5, "Case 5: 3x4 with free variables")

A6 = Matrix.from_lists([[1, 1, 1], [1, 1, 1], [1, 1, 1]])
b6 = [1, 2, 1]
m6 = analyze_system(A6, b6, "Case 6: 3x3 inconsistent (all same row, different b)")

all_match = all([m1, m2, m3, m4, m5, m6])
print(f"All rank predictions match solve(): {all_match}")

---

## 4. Verify — Systematic Checks

We now run thorough verification to confirm that all our implementations are
internally consistent and agree with NumPy.

In [ ]:
print("=" * 60)
print("VERIFICATION 1: Rank-Nullity on Diverse Matrices")
print("=" * 60)
print()

import random
random.seed(42)

all_pass = True
for trial in range(10):
    rows = random.randint(2, 5)
    cols = random.randint(2, 6)
    data = [random.uniform(-10, 10) for _ in range(rows * cols)]
    M = Matrix(rows, cols, data)
    r = rank(M)
    nu = nullity(M)
    holds = (r + nu == cols)
    if not holds:
        all_pass = False
    print(f"  Trial {trial+1} ({rows}x{cols}): rank={r}, nullity={nu}, n={cols}  |  {r}+{nu}={r+nu}  ({'OK' if holds else 'FAIL'})")

print(f"\nAll Rank-Nullity checks passed: {all_pass}")

In [ ]:
print("=" * 60)
print("VERIFICATION 2: Null Space Vectors Satisfy Ax = 0")
print("=" * 60)
print()

test_mats = [
    ("3x4 rank 2", Matrix.from_lists([[1, 2, 3, 4], [2, 4, 7, 10], [3, 6, 10, 14]])),
    ("3x3 rank 2", Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 9]])),
    ("2x3 rank 1", Matrix.from_lists([[1, 2, 3], [2, 4, 6]])),
    ("3x3 rank 3", Matrix.from_lists([[1, 0, 1], [0, 1, 1], [1, 1, 0]])),
]

all_pass = True
for name, M in test_mats:
    ns = null_space(M)
    print(f"{name}: {len(ns)} null space vector(s)")
    for i, v in enumerate(ns):
        x = Matrix.from_vector(v)
        Ax = multiply(M, x)
        result = [Ax[j, 0] for j in range(Ax.rows)]
        is_zero = all(abs(val) < EPSILON for val in result)
        if not is_zero:
            all_pass = False
        print(f"    n{i+1}: Ax = {[round(val, 10) for val in result]}  (zero? {is_zero})")
    print()

print(f"All null space vectors satisfy Ax=0: {all_pass}")

In [ ]:
print("=" * 60)
print("VERIFICATION 3: Column Space Vectors Are Actual Columns of A")
print("=" * 60)
print()

all_pass = True
for name, M in test_mats:
    cs = column_space(M)
    all_cols = [M.get_col(j) for j in range(M.cols)]
    print(f"{name}: column space basis has {len(cs)} vector(s)")
    for i, v in enumerate(cs):
        found = any(all(abs(a - b) < EPSILON for a, b in zip(v, col)) for col in all_cols)
        if not found:
            all_pass = False
        print(f"    c{i+1} = {v}  (is original column? {found})")
    print()

print(f"All column space basis vectors are original columns of A: {all_pass}")

In [ ]:
print("=" * 60)
print("VERIFICATION 4: Compare rank vs np.linalg.matrix_rank")
print("=" * 60)
print()

all_pass = True
for name, M in test_mats:
    our_rank = rank(M)
    np_A = np.array(M.to_lists())
    np_rank = int(np.linalg.matrix_rank(np_A))
    match = (our_rank == np_rank)
    if not match:
        all_pass = False
    print(f"{name}: our rank={our_rank}, numpy rank={np_rank}  ({'OK' if match else 'MISMATCH'})")

print()

random.seed(99)
for trial in range(8):
    rows = random.randint(2, 6)
    cols = random.randint(2, 6)
    data = [random.uniform(-10, 10) for _ in range(rows * cols)]
    M = Matrix(rows, cols, data)
    our_rank = rank(M)
    np_A = np.array(M.to_lists())
    np_rank = int(np.linalg.matrix_rank(np_A))
    match = (our_rank == np_rank)
    if not match:
        all_pass = False
    print(f"Random {rows}x{cols} (trial {trial+1}): our rank={our_rank}, numpy rank={np_rank}  ({'OK' if match else 'MISMATCH'})")

print(f"\nAll rank values match NumPy: {all_pass}")

In [ ]:
print("=" * 60)
print("VERIFICATION 5: Null Space Basis Is Linearly Independent")
print("=" * 60)
print()

all_pass = True
for name, M in test_mats:
    ns = null_space(M)
    if len(ns) == 0:
        print(f"{name}: null space is trivial (empty basis) — OK")
    elif len(ns) == 1:
        is_nonzero = any(abs(x) > EPSILON for x in ns[0])
        print(f"{name}: single null space vector, nonzero? {is_nonzero}")
        if not is_nonzero:
            all_pass = False
    else:
        indep, dep = is_independent(ns)
        print(f"{name}: {len(ns)} null space vectors, independent? {indep}")
        if not indep:
            all_pass = False

print(f"\nAll null space bases are linearly independent: {all_pass}")

---

## 5. Visualize — Column Space and Null Space in $\mathbb{R}^3$

For a $3 \times 3$ matrix, the column space and null space live in $\mathbb{R}^3$.
When $\operatorname{rank} = 2$, the column space is a **plane** through the origin
and the null space is a **line** through the origin. They are complementary:
every vector in $\mathbb{R}^3$ can be decomposed into a part in each.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

B = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("B ="); print(B)
print(f"rank(B) = {rank(B)}, nullity(B) = {nullity(B)}")

cs_B = column_space(B)
ns_B = null_space(B)

print(f"\nColumn space basis: {cs_B}")
print(f"Null space basis:   {ns_B}")

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

c1, c2 = np.array(cs_B[0]), np.array(cs_B[1])
s_vals = np.linspace(-1.5, 1.5, 5)
t_vals = np.linspace(-1.5, 1.5, 5)
ss, tt = np.meshgrid(s_vals, t_vals)
plane_x = c1[0] * ss + c2[0] * tt
plane_y = c1[1] * ss + c2[1] * tt
plane_z = c1[2] * ss + c2[2] * tt

ax.plot_surface(plane_x, plane_y, plane_z, alpha=0.25, color='steelblue',
                edgecolor='steelblue', linewidth=0.3)

for i, col_vec in enumerate(cs_B):
    v = np.array(col_vec)
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color='blue', arrow_length_ratio=0.08,
              linewidth=2.5)
    ax.text(v[0]*1.12, v[1]*1.12, v[2]*1.12, f'c{i+1}', color='blue', fontsize=12,
            fontweight='bold')

n1 = np.array(ns_B[0])
for t_param in np.linspace(-2, 2, 50):
    pt = t_param * n1
    ax.scatter(*pt, color='red', s=3, alpha=0.5)

ax.quiver(0, 0, 0, n1[0], n1[1], n1[2], color='red', arrow_length_ratio=0.1,
          linewidth=2.5)
ax.text(n1[0]*1.3, n1[1]*1.3, n1[2]*1.3, 'n1', color='red', fontsize=12,
        fontweight='bold')

ax.scatter(0, 0, 0, color='black', s=60, zorder=5)
ax.text(0.2, 0.2, 0.2, 'origin', fontsize=9)

ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_zlabel('$x_3$', fontsize=12)
ax.set_title('Column Space (blue plane) and Null Space (red line)\n'
             'for $B$ with rank 2, nullity 1', fontsize=13)

blue_patch = plt.Line2D([0], [0], color='steelblue', linewidth=6, alpha=0.5,
                        label='Col(B) — plane')
red_patch = plt.Line2D([0], [0], color='red', linewidth=3,
                       label='Null(B) — line')
ax.legend(handles=[blue_patch, red_patch], loc='upper left', fontsize=10)

ax.view_init(elev=20, azim=135)
plt.tight_layout()
plt.show()

print("\nThe plane (column space, rank 2) and the line (null space, nullity 1)")
print("together account for all 3 dimensions: 2 + 1 = 3. This is Rank-Nullity.")

---

## 6. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Compute the rank and nullity of the matrix

$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{bmatrix}$$

Then verify the Rank-Nullity Theorem: $\operatorname{rank}(A) + \operatorname{nullity}(A) = 3$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

For the $3 \times 4$ matrix

$$A = \begin{bmatrix} 1 & 0 & 2 & 1 \\ 2 & 1 & 3 & 0 \\ 0 & 1 & -1 & -2 \end{bmatrix}$$

find:
1. A basis for the **row space** using `row_space(A)`
2. A basis for the **column space** using `column_space(A)`
3. A basis for the **null space** using `null_space(A)`
4. Verify rank + nullity = 4
5. Verify each null space vector satisfies $Ax = 0$

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `solution_analysis(A_lists, b_list)` that takes a matrix (as a list of lists)
and a right-hand side vector, and returns a dictionary with:

- `"rank_A"`: rank of $A$
- `"rank_aug"`: rank of $[A \mid b]$
- `"nullity"`: nullity of $A$
- `"type"`: one of `"unique"`, `"infinite"`, `"inconsistent"`
- `"solution"`: the solution vector (unique case), particular solution (infinite case), or `None`
- `"null_basis"`: basis for the null space of $A$

Test it on at least three systems (one of each type). For the infinite case, verify that
`particular + t * null_vector` is also a solution for several values of `t`.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Definition | Computation |
|---|---|---|
| Row space | $\operatorname{span}$ of the rows of $A$ | Nonzero rows of RREF |
| Column space | $\operatorname{span}$ of the columns of $A$ | Original columns at pivot positions |
| Null space | $\{x : Ax = 0\}$ | One basis vector per free variable |
| Rank | $\dim(\operatorname{Row}) = \dim(\operatorname{Col})$ | Number of pivots in RREF |
| Nullity | $\dim(\operatorname{Null})$ | Number of free variables |

**The Rank-Nullity Theorem:**
$$\operatorname{rank}(A) + \operatorname{nullity}(A) = n \quad \text{(number of columns)}$$

**Rank and solvability of $Ax = b$:**

| $\operatorname{rank}(A)$ vs $\operatorname{rank}([A|b])$ | Free variables? | Result |
|---|---|---|
| Equal, $= n$ | None | Unique solution |
| Equal, $< n$ | Yes | Infinitely many solutions |
| Not equal | N/A | No solution |

**Takeaway:** Rank is the fundamental invariant that connects the geometry of subspaces
(row, column, null) to the algebra of linear systems. The Rank-Nullity Theorem
guarantees that every input dimension is accounted for — either $A$ maps it to something
nonzero (rank) or collapses it to zero (nullity).